# Lab type: debug
# Course: ML302 — Transformer Models & Fine-Tuning
# Lesson: Building a Fine-tuning Pipeline
# Task: The pipeline below contains 3 bugs. All three allow the code to run to completion without raising an error, but produce wrong results — either a KeyError at training time, no evaluation during training, or a nonsensical accuracy score. Find each bug, explain it in the markdown cell below it, and fix it.

In [ ]:
# Install dependencies (uncomment if running on Colab)
# !pip install transformers datasets evaluate

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset
import numpy as np

print('Setup complete.')

In [ ]:
# Shared setup: tokeniser, model, raw dataset
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

# AG News: 4-class news topic classification (World, Sports, Business, Sci/Tech)
raw_dataset = load_dataset('fancyzhx/ag_news')
# Use a small slice for fast iteration in the lab
train_small = raw_dataset['train'].select(range(400))
test_small  = raw_dataset['test'].select(range(100))

print('Train size:', len(train_small))
print('Test  size:', len(test_small))
print('Columns:   ', train_small.column_names)
print('Sample:    ', train_small[0])

## Bug 1: Missing column rename and text column not removed

The tokenisation function is correct, but the `map` call omits two required post-processing steps. The dataset is passed to the Trainer, which will raise a `KeyError` when it tries to access the `labels` column (which doesn't exist — the raw column is named `label`). The `text` column will also cause a tensor collation error.

Run the cell below and read the output carefully — it does not raise an error yet, but will fail at `trainer.train()`.

In [ ]:
# Bug 1 — tokenise without fixing column names or removing non-tensor columns
def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized_bug1 = train_small.map(tokenize_fn, batched=True)
tokenized_bug1.set_format('torch')

print('Columns after map:', tokenized_bug1.column_names)
print()
print('NOTE: the column is named "label" (singular), not "labels" (plural).')
print('The Trainer looks for "labels". It will raise KeyError when building a batch.')
print('Also, "text" column is still present — strings cannot be collated into tensors.')

**Explain Bug 1:** The Trainer calls `DataCollatorWithPadding`, which tries to stack all columns into tensors. Explain why two specific missing steps cause failures, and state what each step achieves.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**Step 1 — rename `label` → `labels`:** HuggingFace models expect the target column to be named exactly `labels`. Without the rename, the Trainer passes no label tensor to the model's forward call, and the model returns only logits with no loss — training silently does nothing useful (or raises a `TypeError` depending on the version).

**Step 2 — remove the `text` column:** `DataCollatorWithPadding` tries to batch every remaining dataset column into a tensor. The `text` column is a list of strings — `torch.tensor` cannot convert strings to a numeric tensor, so the collator throws a `ValueError` at the first training step. Removing it before training prevents the collator from seeing any non-numeric columns.

**How to catch early:** Inspect `tokenized_dataset.features` and `tokenized_dataset.column_names` before calling `Trainer`. Verify that `labels` is present, `text` is absent, and all remaining columns are numeric (`input_ids`, `attention_mask`, `token_type_ids`).

</details>

In [ ]:
# Fix Bug 1: rename 'label' -> 'labels', remove 'text'
tokenized_fixed = train_small.map(tokenize_fn, batched=True)
tokenized_fixed = tokenized_fixed.rename_column('label', 'labels')  # required key
tokenized_fixed = tokenized_fixed.remove_columns(['text'])          # cannot collate strings
tokenized_fixed.set_format('torch')

test_tokenized = test_small.map(tokenize_fn, batched=True)
test_tokenized = test_tokenized.rename_column('label', 'labels')
test_tokenized = test_tokenized.remove_columns(['text'])
test_tokenized.set_format('torch')

print('Columns after fix:', tokenized_fixed.column_names)
print('Sample keys:', list(tokenized_fixed[0].keys()))

## Bug 2: `evaluation_strategy='no'` — no evaluation during training

The `TrainingArguments` below set `evaluation_strategy='no'`. This is the Trainer default. Training runs to completion, loss decreases, and no error is raised — but no validation metrics are logged, no best model checkpoint is saved, and overfitting cannot be detected.

In [ ]:
# Bug 2 — evaluation_strategy='no' (the Trainer default)
training_args_bug2 = TrainingArguments(
    output_dir='./results_bug2',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy='no',  # BUG: training runs blind — no val metrics
    logging_steps=20,
    report_to='none',
)

# Minimal trainer to show logging output (no compute_metrics yet)
trainer_bug2 = Trainer(
    model=model,
    args=training_args_bug2,
    train_dataset=tokenized_fixed,
    eval_dataset=test_tokenized,
)
# trainer_bug2.train()  # Uncomment to run — observe: no eval logs appear
print('evaluation_strategy:', training_args_bug2.evaluation_strategy)
print('NOTE: No evaluation will run during training. The eval_dataset is ignored.')
print('Result: overfitting is undetectable until training is already complete.')

**Explain Bug 2:** In a full fine-tuning run (not this lab slice), `evaluation_strategy='no'` means the Trainer never calls `evaluate()` during training. State two concrete consequences: one related to model selection, one related to detecting overfitting.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**Model selection consequence:** With `evaluation_strategy='no'`, there is no validation loss signal, so `load_best_model_at_end=True` has nothing to compare — the final checkpoint (which may be the most overfit) is saved as the "best" model. You deploy the epoch-N checkpoint rather than the one with the lowest validation loss.

**Overfitting detection consequence:** Without per-epoch evaluation, training loss decreasing while validation accuracy plateaus (the classic overfitting signature) is invisible. You only discover the problem after training completes, wasting compute. With `evaluation_strategy='epoch'`, you can add early stopping (`EarlyStoppingCallback`) to halt training the moment validation performance stops improving.

</details>

In [ ]:
# Fix Bug 2: set evaluation_strategy='epoch' and enable best model checkpointing
training_args_fixed = TrainingArguments(
    output_dir='./results_fixed',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy='epoch',       # evaluate after every epoch
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=20,
    report_to='none',
)
print('evaluation_strategy:', training_args_fixed.evaluation_strategy)
print('load_best_model_at_end:', training_args_fixed.load_best_model_at_end)

## Bug 3: `compute_metrics` compares raw logits to class indices

The `compute_metrics` function below looks plausible — it unpacks `eval_pred` and computes accuracy. But `logits` are floating-point arrays of shape `(n_examples, n_classes)`, not class indices. Comparing them to integer labels with `==` will almost never produce True.

In [ ]:
# Bug 3 — logits not converted to class indices before comparison
def compute_metrics_bug3(eval_pred):
    logits, labels = eval_pred
    # BUG: logits shape is (n, 4) — floats. labels are integers 0-3.
    # This comparison is almost always False, producing accuracy near 0.0
    return {'accuracy': float((logits == labels).mean())}

# Demonstrate the bug on synthetic data
fake_logits = np.array([[2.1, -0.3, 0.5, -1.2], [-0.8, 3.4, 0.1, 0.9]])
fake_labels = np.array([0, 1])

result_buggy = compute_metrics_bug3((fake_logits, fake_labels))
print('Buggy accuracy:', result_buggy)
# → accuracy: 0.0 (or near-zero) even though the predictions are correct!

correct_preds = np.argmax(fake_logits, axis=-1)
print('Argmax predictions:', correct_preds)
print('True labels:       ', fake_labels)
print('Manual check: are predictions correct?', (correct_preds == fake_labels).all())

**Explain Bug 3:** The logits for a 4-class classifier have shape `(n, 4)`. The true labels are integers in `{0, 1, 2, 3}`. Explain exactly why `logits == labels` evaluates to near-zero accuracy even when the model's predictions are entirely correct. What is the correct conversion step?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**Why `logits == labels` is always wrong:** `logits` has shape `(n, 4)` — each row is a vector of four real-valued scores, one per class. `labels` has shape `(n,)` — each value is an integer in `{0, 1, 2, 3}`. NumPy (and PyTorch) will broadcast `(n, 4) == (n,)` element-wise in a way that virtually never produces `True`, because you are comparing a float score (e.g., `2.35`) to a class index (e.g., `2`). Even if logit value 2.35 happened to round to 2, you would be comparing the wrong thing — the *magnitude* of the score is not the predicted class.

**Correct conversion:** Apply `np.argmax(logits, axis=-1)` (or `logits.argmax(-1)`) to reduce `(n, 4)` → `(n,)` by taking the index of the highest score for each example. Then compare `predictions == labels` — both are integer class indices in `{0, 1, 2, 3}`.

</details>

In [ ]:
# Fix Bug 3: apply argmax before comparing to labels
def compute_metrics_fixed(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)  # (n,) integer class indices
    return {'accuracy': float((predictions == labels).mean())}

result_fixed = compute_metrics_fixed((fake_logits, fake_labels))
print('Fixed accuracy:', result_fixed)  # → {'accuracy': 1.0}

# Full trainer with all three bugs fixed
from transformers import AutoModelForSequenceClassification
model_clean = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)

trainer_full = Trainer(
    model=model_clean,
    args=training_args_fixed,
    train_dataset=tokenized_fixed,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics_fixed,
)
# trainer_full.train()  # Uncomment to run a full (slow) training pass
print('Trainer configured. Uncomment trainer_full.train() to run.')

## Summary

> **For each bug, write one sentence on what went wrong and how to catch it early.**

1. Missing column rename / text removal: 
2. `evaluation_strategy='no'`: 
3. Logits not argmax'd in `compute_metrics`: 

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Missing column rename / text removal:** Forgetting to rename `label` → `labels` silently suppresses the loss, and leaving the `text` column causes the data collator to crash on the first batch; catch it early by printing `dataset.column_names` and `dataset.features` before calling `Trainer`.

2. **`evaluation_strategy='no'`:** Without per-epoch evaluation, overfitting is invisible during training and the final checkpoint (not the best one) is saved; catch it early by setting `evaluation_strategy='epoch'` and checking that validation metrics are logged in `trainer.state.log_history`.

3. **Logits not argmax'd in `compute_metrics`:** Comparing the raw `(n, 4)` logit matrix directly to `(n,)` class indices produces nonsensical accuracy near zero even for a well-trained model; catch it early by asserting `predictions.shape == labels.shape` at the top of `compute_metrics`.

</details>